# Customer Complaint Classification & Routing Engine

## AI Agent with FastAPI, vLLM, PydanticAI, Vector DB & Live Routing Dashboard

This notebook demonstrates how to build an enterprise-grade AI complaint routing system.

Features:
- Complaint classification
- Priority detection
- Sentiment analysis
- Team routing
- SLA awareness
- Vector database storage
- Real-time dashboard
- FastAPI endpoints
- MCP tool integration


## Step 1: Install Dependencies

In [ ]:
!pip install -q pydantic_ai openai fastapi uvicorn streamlit chromadb sentence-transformers pandas plotly

## Step 2: Configure Environment

In [ ]:
import os

BASE_URL = "http://localhost:8000/v1"

os.environ["BASE_URL"] = BASE_URL
os.environ["OPENAI_API_KEY"] = "abc-123"

print("Environment Configured")

## Step 3: Initialize OpenAI-Compatible Model

In [ ]:
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"]
)

agent_model = OpenAIModel(
    "complaint-agent",
    provider=provider
)

print("Model Initialized")

## Step 4: Create AI Agent

In [ ]:
from pydantic_ai import Agent

agent = Agent(
    model=agent_model
)

print("Agent Created")

## Step 5: Helper Function

In [ ]:
import asyncio

async def run_async(prompt):
    result = await agent.run(prompt)
    return result.output

## Step 6: Classification Tool

In [ ]:
from pydantic_ai import Tool

@Tool
def classify_complaint(text: str) -> dict:
    """Classify customer complaint"""

    categories = {
        "billing": ["invoice", "charged", "payment"],
        "technical": ["error", "issue", "bug", "internet"],
        "refund": ["refund", "return"]
    }

    text_lower = text.lower()

    for category, keywords in categories.items():
        if any(k in text_lower for k in keywords):
            return {"category": category}

    return {"category": "general"}

## Step 7: Priority Detection Tool

In [ ]:
@Tool
def detect_priority(text: str):

    urgent_keywords = [
        "angry",
        "cancel",
        "legal",
        "urgent",
        "immediately"
    ]

    score = 1

    for word in urgent_keywords:
        if word in text.lower():
            score += 2

    if score >= 5:
        priority = "P1"
    elif score >= 3:
        priority = "P2"
    else:
        priority = "P3"

    return {
        "priority": priority,
        "score": score
    }

## Step 8: Sentiment Analysis Tool

In [ ]:
@Tool
def analyze_sentiment(text: str):

    negative_words = [
        "bad",
        "terrible",
        "angry",
        "worst"
    ]

    matches = sum(
        word in text.lower()
        for word in negative_words
    )

    return {
        "sentiment": "negative" if matches > 1 else "neutral"
    }

## Step 9: Update Agent with Tools

In [ ]:
agent = Agent(
    model=agent_model,
    tools=[
        classify_complaint,
        detect_priority,
        analyze_sentiment
    ],
    system_prompt="""
You are an enterprise complaint routing AI.

Tasks:
1. Classify complaint
2. Detect severity
3. Analyze sentiment
4. Route to proper team
5. Predict SLA urgency
"""
)

print("Enhanced Agent Ready")

## Step 10: Test Complaint Classification

In [ ]:
response = await run_async(
    "My internet is down for 2 days and this is terrible service"
)

print(response)

## Step 11: FastAPI Complaint API

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Complaint(BaseModel):
    customer_id: str
    message: str

@app.post("/submit")
async def submit_complaint(c: Complaint):

    result = await agent.run(
        f"Analyze complaint: {c.message}"
    )

    return result.output

## Step 12: Run FastAPI Server

In [ ]:
# Run this in terminal
# uvicorn app:app --reload

## Step 13: Setup ChromaDB

In [ ]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="complaints"
)

print("Vector DB Ready")

## Step 14: Store Complaint Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

complaint_text = "Internet outage for 2 days"

embedding = embedding_model.encode(
    complaint_text
).tolist()

collection.add(
    documents=[complaint_text],
    embeddings=[embedding],
    ids=["case_001"]
)

print("Complaint Stored")

## Step 15: Similarity Search

In [ ]:
results = collection.query(
    query_embeddings=[embedding],
    n_results=3
)

print(results)

## Step 16: Dashboard Code

In [ ]:
dashboard_code = """
import streamlit as st
import pandas as pd

st.title('Complaint Routing Dashboard')

df = pd.read_csv('complaints.csv')

st.bar_chart(df['category'].value_counts())

st.dataframe(df)
"""

print(dashboard_code)

## Step 17: SLA Escalation Logic

In [ ]:
from datetime import datetime, timedelta

def check_sla(ticket):

    elapsed = datetime.now() - ticket["created"]

    if ticket["priority"] == "P1":
        return elapsed.seconds > 3600

    return False

ticket = {
    "priority": "P1",
    "created": datetime.now() - timedelta(hours=2)
}

print(check_sla(ticket))

## Step 18: Kafka Producer Example

In [ ]:
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers='localhost:9092'
)

producer.send(
    'complaints',
    b'internet not working'
)

## Step 19: Kafka Consumer Example

In [ ]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    'complaints',
    bootstrap_servers='localhost:9092'
)

for msg in consumer:
    print(msg.value)

## Step 20: Dockerfile

In [ ]:
dockerfile = """
FROM python:3.11

WORKDIR /app

COPY . .

RUN pip install -r requirements.txt

CMD ["uvicorn", "app:app", "--host", "0.0.0.0"]
"""

print(dockerfile)

## Step 21: Final Demo Flow

In [ ]:
demo_flow = """
Incoming Complaint
        ↓
AI Classification
        ↓
Priority Detection
        ↓
Sentiment Analysis
        ↓
Team Routing
        ↓
SLA Validation
        ↓
Dashboard Monitoring
"""

print(demo_flow)